# 01 — Análise Exploratória de Dados (EDA)

**Objetivo:** Entender a distribuição das variáveis, identificar valores ausentes e outliers,
explorar correlações com a variável-alvo (`No-show`) e formular hipóteses iniciais
sobre os fatores que influenciam a falta em consultas do SUS.

## 1. Primeira leitura do dataset (issue #3)

Visão inicial de cada coluna: tipo, significado, valores e problemas.
O resumo está em [`docs/dicionario_dados.md`](../docs/dicionario_dados.md).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/raw/KaggleV2-May-2016.csv')
print(df.shape)
df.head()

### 1.1 Tipos e valores ausentes

In [ ]:
df.info()

In [ ]:
print('Nulos:', df.isna().sum().sum())
print('Linhas duplicadas:', df.duplicated().sum())
print('AppointmentID repetido:', df['AppointmentID'].duplicated().sum())
print('Duplicadas ignorando AppointmentID:', df.drop(columns='AppointmentID').duplicated().sum())

Não há nulos nem linhas idênticas, mas 618 linhas repetem tudo menos o `AppointmentID`.
Em alguns desses grupos o alvo diverge:

In [ ]:
chave = [c for c in df.columns if c not in ('AppointmentID', 'No-show')]
print('Grupos duplicados com alvo diferente:', (df.groupby(chave)['No-show'].nunique() > 1).sum())

### 1.2 Identificadores

In [ ]:
print('Pacientes distintos:', df['PatientId'].nunique())
print('Pacientes com mais de um agendamento:', (df['PatientId'].value_counts() > 1).sum())
print('PatientId com casas decimais:', (df['PatientId'] % 1 != 0).sum())
df.loc[df['PatientId'] % 1 != 0, 'PatientId']

In [ ]:
# AppointmentID cresce junto com a data de marcação: carrega informação temporal, não usar como feature
sched = pd.to_datetime(df['ScheduledDay'])
appt = pd.to_datetime(df['AppointmentDay'])
print('Correlação AppointmentID x ScheduledDay:', round(df['AppointmentID'].corr(sched.astype('int64')), 4))

### 1.3 Variável-alvo `No-show` (`Yes` = faltou)

In [ ]:
print(df['No-show'].value_counts())
print(df['No-show'].value_counts(normalize=True).round(3))

### 1.4 Variáveis categóricas e binárias

In [ ]:
falta = df['No-show'].eq('Yes')
for col in ['Gender', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received']:
    resumo = pd.DataFrame({'n': df[col].value_counts(), 'taxa_falta': falta.groupby(df[col]).mean().round(3)})
    print(f'--- {col}'); print(resumo, end='\n\n')

`Handcap` não é binária: é o número de deficiências (0 a 4).
`Scholarship` indica inscrição no Bolsa Família, não bolsa de estudos.

### 1.5 Idade

In [ ]:
print(df['Age'].describe())
print('Idade negativa:', (df['Age'] < 0).sum())
print('Idade zero:', (df['Age'] == 0).sum())
df.loc[df['Age'] > 100, ['PatientId', 'Age', 'Gender', 'AppointmentDay']]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
sns.histplot(df['Age'], bins=60, ax=ax)
ax.set(title='Distribuição de idade', xlabel='Idade', ylabel='Agendamentos')
plt.show()

### 1.6 Bairro

In [ ]:
bairros = pd.DataFrame({'n': df['Neighbourhood'].value_counts(),
                        'taxa_falta': falta.groupby(df['Neighbourhood']).mean().round(3)})
print('Bairros distintos:', len(bairros))
print('Bairros com menos de 50 agendamentos:')
bairros[bairros['n'] < 50]

### 1.7 Datas e dias de espera

In [ ]:
print('ScheduledDay:', sched.min(), '->', sched.max())
print('AppointmentDay:', appt.min(), '->', appt.max())
print('Horas distintas em AppointmentDay:', appt.dt.strftime('%H:%M:%S').unique())
print('Dias com consulta:', appt.dt.normalize().nunique())
appt.dt.day_name().value_counts()

In [ ]:
espera = (appt.dt.normalize() - sched.dt.normalize()).dt.days
print(espera.describe())
print('Espera negativa:', (espera < 0).sum(), espera[espera < 0].value_counts().to_dict())
print('Mesmo dia:', (espera == 0).sum())
print('Taxa de falta (mesmo dia):', round(falta[espera == 0].mean(), 3))
print('Taxa de falta (com espera):', round(falta[espera > 0].mean(), 3))

### 1.8 A armadilha do SMS

Quem recebeu SMS parece faltar mais. Mas o SMS só vai para consultas com 3 ou mais dias de espera,
que já faltam mais. Comparando dentro desse grupo, o efeito se inverte.

In [ ]:
print('SMS em consultas com espera <= 2 dias:', df.loc[espera <= 2, 'SMS_received'].sum())
print('Taxa de falta por SMS (todos):', falta.groupby(df['SMS_received']).mean().round(3).to_dict())
m = espera >= 3
print('Taxa de falta por SMS (espera >= 3 dias):', falta[m].groupby(df.loc[m, 'SMS_received']).mean().round(3).to_dict())

### 1.9 Resumo dos problemas

| Problema | Linhas | O que fazer (S2) |
|---|---|---|
| Dias de espera negativos | 5 | Remover |
| Idade negativa | 1 | Remover |
| Idade acima de 100 | 7 | Avaliar (4 são do mesmo paciente com 115) |
| Duplicatas sem `AppointmentID` | 618 | Decidir se remove; 14 grupos têm alvo divergente |
| `PatientId` como float, 5 com decimais | 5 | Converter para string |
| Datas como texto | todas | Converter para datetime |
| Nomes com erro / hífen | — | Renomear (`Hipertension`, `Handcap`, `No-show`) |
| `Handcap` é contagem, não binária | — | Manter como inteiro ou binarizar com cuidado |
| `SMS_received` confundido com espera | — | Controlar pelo tempo de espera nas análises |
| Bairros com poucos registros | ~7 | Agrupar em "outros" na modelagem |

## 2. EDA agregada no conjunto de treino (S3 — issue #11)

Diferente da seção 1 — primeira leitura do CSV **bruto**, antes da limpeza —, esta seção
lê **tabelas agregadas congeladas** geradas por `src.eda` somente sobre a partição de
**treino** (divisão por paciente, seed 42): 87.720 agendamentos de 49.838 pacientes.
O teste (22.169 agendamentos) permanece reservado e não entra em nenhuma conta aqui.
O alvo já está codificado: `no_show` = 1 significa que o paciente **faltou**.

Cada tabela traz `n` (agendamentos), `no_shows` (faltas), `no_show_rate` (taxa de falta)
e `share` (participação no treino). São **associações descritivas**, não efeitos causais.
Bairros com poucas consultas não sustentam taxas confiáveis. O `sms_received` é posterior
ao agendamento e fica confundido com a espera — por isso foi **excluído dos preditores**
(ver `docs/decisoes.md`); aparece aqui apenas como descrição.

Reprodução (não executada neste notebook):

```bash
python -m src.eda           # gera reports/sprints-3-4/eda/*.csv e summary.json
python -m src.sprint_plots  # gera reports/sprints-3-4/figures/*.png
```

In [ ]:
import sys
import json
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display, Image

EDA = ROOT / 'reports/sprints-3-4/eda'
FIGURAS = ROOT / 'reports/sprints-3-4/figures'
summary = json.loads((EDA / 'summary.json').read_text())
{chave: summary[chave] for chave in
 ('clean_rows', 'eda_rows', 'reserved_test_rows', 'patients', 'no_shows', 'no_show_rate')}

### 2.1 Tabelas agregadas

In [ ]:
for nome in ['age_group', 'gender', 'neighbourhood', 'bolsa_familia',
             'hypertension', 'diabetes', 'alcoholism', 'handicap',
             'waiting_group', 'weekday', 'sms_received', 'sms_by_waiting']:
    print(f'--- {nome}')
    display(pd.read_csv(EDA / f'{nome}.csv'))

### 2.2 Figuras

Barras: número de agendamentos. Linha/marcadores: taxa de falta (%).

In [ ]:
for png in sorted(FIGURAS.glob('eda_*.png')):
    display(Image(filename=str(png)))

### 2.3 Leitura dos resultados

Na amostra de treino (87.720 agendamentos; 17.594 faltas), a taxa geral é 20,06%.
Consultas no mesmo dia têm taxa de 4,61%, contra 33,99% na faixa de 31–60 dias; a
relação não é estritamente crescente (61+: 29,20%). A faixa de 13–17 anos apresenta
26,57%, e 60+ apresenta 15,21%. São associações descritivas, não efeitos causais.
O SMS aparece associado a mais faltas no agregado (27,43% com SMS; 16,54% sem), mas a
menos faltas dentro de cada faixa de espera observada a partir de 3 dias. Não há
comparação com SMS nas faixas de até 2 dias. Os estratos largos ainda podem ter
confundimento residual; isso não estima eficácia do SMS. Bairros extremos com poucos
registros, como 2 faltas em 2 consultas, não sustentam uma classificação confiável de
risco por bairro.